# 01.2 — Data audit

Mục tiêu: đo các vấn đề quan sát được trong RAW. Câu hỏi: lỗi nào đủ chắc chắn để loại tự động? Trả lời: chỉ non-string, blank, control/replacement character và duplicate chính xác; các tín hiệu còn lại được báo cáo để kiểm tra.

In [1]:
from pathlib import Path
import re, unicodedata, json
import pandas as pd

ROOT = Path.cwd().resolve()
while not (ROOT / 'configs').is_dir(): ROOT = ROOT.parent
config = json.loads((ROOT / 'configs/data/phase01_it_en_vi.json').read_text(encoding='utf-8'))
frames = []
for source, spec in config['sources'].items():
    raw = pd.read_parquet(ROOT / 'data/raw' / source / f'{source}_raw.parquet')
    frames.append(pd.DataFrame({'source': source, 'en': raw[spec['en']], 'vi': raw[spec['vi']]}))
raw_pairs = pd.concat(frames, ignore_index=True)
SPACE = re.compile(r'\s+')
def normalize(value): return SPACE.sub(' ', unicodedata.normalize('NFC', value)).strip()


In [2]:
is_text = raw_pairs.en.map(lambda x: isinstance(x, str)) & raw_pairs.vi.map(lambda x: isinstance(x, str))
safe_en = raw_pairs.en.fillna('').map(str)
safe_vi = raw_pairs.vi.fillna('').map(str)
en = safe_en.map(normalize); vi = safe_vi.map(normalize)
ratio = en.str.len() / vi.str.len().clip(lower=1)
CONTROL = re.compile(r'[\x00-\x08\x0b\x0c\x0e-\x1f\x7f-\x9f]')
broken_text = safe_en.map(lambda value: '\ufffd' in value or bool(CONTROL.search(value))) | safe_vi.map(lambda value: '\ufffd' in value or bool(CONTROL.search(value)))
issues = [
    ('Non-string pair', (~is_text).sum()),
    ('Null or blank pair', (is_text & ((safe_en.str.strip() == '') | (safe_vi.str.strip() == ''))).sum()),
    ('Control/replacement character', broken_text.sum()),
    ('Identical EN–VI', (is_text & (en == vi)).sum()),
    ('Length imbalance (>3.5×)', (is_text & ((ratio > 3.5) | (ratio < 1/3.5))).sum()),
    ('Very long pair (>1000 chars)', (is_text & ((en.str.len() > 1000) | (vi.str.len() > 1000))).sum()),
    ('Exact duplicate in RAW', raw_pairs.assign(en_norm=en, vi_norm=vi).duplicated(['en_norm', 'vi_norm']).sum()),
]
audit_summary = pd.DataFrame(issues, columns=['issue_type', 'count'])
audit_summary['rate_percent'] = (100 * audit_summary['count'] / len(raw_pairs)).round(3)
audit_summary

,issue_type,count,rate_percent
0,Non-string pair,0,0.000
1,Null or blank pair,0,0.000
2,Control/replacement character,6,0.004
3,Identical EN–VI,2446,1.493
4,Length imbalance (>3.5×),3232,1.972
5,Very long pair (>1000 chars),29,0.018
6,Exact duplicate in RAW,4773,2.913


In [3]:
abnormal_ratio = raw_pairs.assign(
    audit_signal=ratio.map(lambda value: 'EN dài hơn VI trên 3.5×' if value > 3.5 else 'VI dài hơn EN trên 3.5×'),
    length_ratio_percent=ratio.map(lambda value: f'{value:.1%}')
).loc[(ratio > 3.5) | (ratio < 1/3.5), ['source', 'en', 'vi', 'audit_signal', 'length_ratio_percent']]
abnormal_ratio.sample(min(5, len(abnormal_ratio)), random_state=42)

,source,en,vi,audit_signal,length_ratio_percent
154089,kde4,Here you can configure the color that is used ...,& Dòng,EN dài hơn VI trên 3.5×,3066.7%
158877,kde4,Table,Ô,EN dài hơn VI trên 3.5×,500.0%
155524,kde4,Inversed Subtract,Trừ,EN dài hơn VI trên 3.5×,566.7%
17497,tech_viet_translation,BlackBerry sells legacy patents related to mob...,None,EN dài hơn VI trên 3.5×,1825.0%
48923,tech_viet_translation,5 simple Apple Watch tips and tricks everyone ...,None,EN dài hơn VI trên 3.5×,1525.0%


In [4]:
from IPython.display import Markdown, display

def show_examples(issue_type, mask, columns=('source', 'en', 'vi'), frame=raw_pairs):
    subset = frame.loc[mask, list(columns)]
    display(Markdown(f'### {issue_type}'))
    if subset.empty:
        print('Không phát hiện ví dụ trong RAW.')
    else:
        display(subset.sample(min(5, len(subset)), random_state=42))

blank_pair = is_text & ((safe_en.str.strip() == '') | (safe_vi.str.strip() == ''))
show_examples('Non-string pair', ~is_text)
show_examples('Null or blank pair', blank_pair)
show_examples('Control/replacement character', broken_text)
show_examples('Identical EN–VI', is_text & (en == vi))
show_examples('Very long pair (>1000 chars)', is_text & ((en.str.len() > 1000) | (vi.str.len() > 1000)))

duplicates = raw_pairs.assign(en_norm=en, vi_norm=vi)
duplicates = duplicates.loc[duplicates.duplicated(['en_norm', 'vi_norm'], keep=False)].copy()
duplicates['duplicate_count'] = duplicates.groupby(['en_norm', 'vi_norm'])['source'].transform('size')
show_examples('Exact duplicate in RAW', pd.Series(True, index=duplicates.index), ('source', 'en', 'vi', 'duplicate_count'), duplicates)


### Non-string pair

Không phát hiện ví dụ trong RAW.


### Null or blank pair

Không phát hiện ví dụ trong RAW.


### Control/replacement character

...

,source,en,vi
162134,kde4,ARCH D,ARCH D
126175,kde4,Leon Pennington,Leon Pennington
123313,kde4,Y:,Y:
163375,kde4,5MB,5MB
119232,ubuntu,Super Lotsa Added Stuff Hack - Extended Magic ...,Super Lotsa Added Stuff Hack - Extended Magic ...


### Very long pair (>1000 chars)

,source,en,vi
160551,kde4,Paper Size This field indicates the paper size...,Kích cỡ giấy Trường này ngụ ý kích cỡ giấy nơi...
149768,kde4,No hide This option is based on the option of ...,Không ẩn Tùy chọn này dựa vào tùy chọn cùng tê...
135342,kde4,Windows Shares Konqueror is able to access sha...,Chia sẻ WindowsTrình Konqueror có khả năng tru...
160401,kde4,List of Driver Options (from PPD). The upper p...,Danh sách tùy chọn trình điều khiển (từ PPD). ...
133893,kde4,File Associations This module allows you to ch...,Liên kết tập tin Môđun này cho phép người dùng...


### Exact duplicate in RAW

,source,en,vi,duplicate_count
121514,kde4,URL:,URL:,2
12525,envitech_reasoning,A blessing in disguise.,Cái rủi có cái may.,7
128423,kde4,Environment Variables,Biến,2
153547,kde4,& New...,& Mới...,9
158155,kde4,Short,Ngắn,3


Các sample bất thường phải được đọc theo ngữ cảnh IT. Một command, path hoặc chuỗi UI không tự động là dữ liệu lỗi.